In [1]:
import gymnasium as gym
import pickle
import numpy as np

## Parameters

In [2]:
alpha = 0.1
gamma = 0.99
epsilon = 1.0 
epsilon_min = 0.01
epsilon_decay = 0.9995
episodes = 7000
nb_disc = 20

We split in 21 to apply discretization

In [3]:
bins_x = np.linspace(-1.0, 1.0, nb_disc)
bins_y = np.linspace(-1.0, 1.0, nb_disc)
bins_vel = np.linspace(-8.0, 8.0, nb_disc)

actions_continues = np.linspace(-2.0, 2.0, 11)
nb_actions = len(actions_continues)

q_table = np.zeros((len(bins_x) + 1, len(bins_y) + 1, len(bins_vel) + 1, nb_actions))


$Q \in \mathbb{R}^{4}$ with $x = 21$, $y = 21$, $\text{vel} = 21$, $N = 11$

## Discretization

In [4]:
def discretize_state(obs):
    x_idx = np.digitize(obs[0], bins_x)
    y_idx = np.digitize(obs[1], bins_y)
    vel_idx = np.digitize(obs[2], bins_vel)
    return (x_idx, y_idx, vel_idx)

## Training

$$Q(s_t, a_t) = (1 - \alpha) \cdot Q(s_t, a_t) + \alpha [R_t + \gamma \max_{a} Q(s_{t+1}, a_t)]$$

In [57]:
env = gym.make("Pendulum-v1", render_mode=None)

for episode in range(episodes):
    obs, info = env.reset()
    state = discretize_state(obs)
    done = False
    truncated = False
    
    while not (done or truncated):
        m = np.random.uniform(0,1)
        if m < epsilon:
            action_idx = np.random.randint(0, nb_actions)
        else:
            action_idx = np.argmax(q_table[state])
            
        # L'environnement s'attend à recevoir une action continue sous forme de ndarray(1,)
        action_continue = np.array([actions_continues[action_idx]], dtype=np.float32)
        
        next_obs, reward, done, truncated, info = env.step(action_continue)
        next_state = discretize_state(next_obs)
        
        best_next_action = np.argmax(q_table[next_state]) # next_state = (i, j, k) so take automatically the maximum value in action values
        q_table[state][action_idx] = (1 - alpha) * q_table[state][action_idx] + alpha * (reward + gamma * q_table[next_state][best_next_action])
        
        state = next_state
        
    # decreasing of epsilon
    if epsilon > epsilon_min:
        epsilon *= epsilon_decay
        
    if (episode + 1) % 500 == 0:
        print(f"Episode {episode + 1}/{episodes} finished.")
        
env.close()

with open("pendulum.pkl", "wb") as f:
    pickle.dump(q_table, f)
print("Training finish")


Episode 500/7000 finished.
Episode 1000/7000 finished.
Episode 1500/7000 finished.
Episode 2000/7000 finished.
Episode 2500/7000 finished.
Episode 3000/7000 finished.
Episode 3500/7000 finished.
Episode 4000/7000 finished.
Episode 4500/7000 finished.
Episode 5000/7000 finished.
Episode 5500/7000 finished.
Episode 6000/7000 finished.
Episode 6500/7000 finished.
Episode 7000/7000 finished.
Training finish


We put a decreasing $\epsilon$ (multiplied by $\epsilon_{\text{decay}} = 0.995$) and with the minimum of $\epsilon_{\text{min}} = 0.01$.

## Testing

In [12]:
with open("pendulum.pkl", "rb") as f:
    q_table_loaded = pickle.load(f)
    
env = gym.make("Pendulum-v1", render_mode="human",max_episode_steps=500)

obs, info = env.reset()
state = discretize_state(obs)
done = False
truncated = False
total_reward = 0

while not (done or truncated):
    action_idx = np.argmax(q_table_loaded[state])
    action_continue = np.array([actions_continues[action_idx]], dtype=np.float32)
    
    next_obs, reward, done, truncated, info = env.step(action_continue)
    state = discretize_state(next_obs)
    total_reward += reward
    
print(f"Total reward : {total_reward:.2f}")
    
env.close()


Total reward : -128.34
